# Fase 2: Evaluación Comparativa y Análisis Estadístico

## 1. Introducción

Una vez establecidos los hiperparámetros óptimos para cada algoritmo (mitigando el sesgo de configuración), esta segunda fase tiene como objetivo evaluar el desempeño general y la escalabilidad del modelo propuesto. Se someterá al algoritmo de **Optimización por Colonia de Hormigas (ACO)** a una comparación de rendimiento directa frente a la **Búsqueda Tabú (TS)**, operando bajo las mismas restricciones teóricas y computacionales.

Para garantizar la generalización de los resultados y evitar el sobreajuste topológico, la experimentación no se limitará a la red base, sino que se extenderá a tres instancias de grafos con distintas complejidades.

---

## 2. Hipótesis de Comparación Algorítmica (H2)

La experimentación de esta fase está guiada por la siguiente hipótesis formal:

> *El algoritmo ACO, al fundamentarse en una exploración paralela y la construcción estigmérgica de memoria a largo plazo, presentará diferencias estadísticamente significativas a favor en la minimización de la función objetivo frente a la Búsqueda Tabú. Esta superioridad se acentuará en redes de mayor densidad topológica, donde la trayectoria única de la Búsqueda Tabú será más propensa a caer en óptimos locales inducidos por callejones sin salida, a pesar de su memoria de corto plazo.*

### Justificación Teórica

* **Búsqueda Tabú (Línea Base):** Es un algoritmo de trayectoria única. Su mecanismo para escapar de mínimos locales es una memoria de corto plazo (Lista Tabú). En redes densas o asimétricas, forzar movimientos subóptimos puede derivar en un alto consumo energético por *deadheading*, consumiendo la batería antes de lograr una exploración significativa.
* **ACO (Modelo Propuesto):** Emplea una población de $m$ hormigas construyendo soluciones en paralelo. La matriz de feromonas actúa como una memoria colectiva distributiva que, balanceada con el factor heurístico local ($\eta_{ij} = 1 / (c_{ij} \cdot \Omega)$), permite esquivar zonas redundantes (callejones) de manera emergente sin depender de un historial rígido de movimientos prohibidos.

---

## 3. Diseño Experimental y Metodología

Para aislar la variabilidad estocástica y obtener validez estadística, se aplicará el siguiente protocolo riguroso:

1. **Banco de Pruebas Inmutable:** Se utilizarán tres redes serializadas para asegurar que ambos algoritmos enfrenten la misma topología:
   * `red_base_5x5.json`: Grafo estándar de control.
   * `red_densa_10x10.json`: Grafo de alta complejidad combinatoria.
   * `red_realista_asim.json`: Grafo con perturbaciones asimétricas simulando un entorno urbano no planificado.

2. **Control Estocástico:**
   Se ejecutarán **15 corridas independientes** por cada configuración (Algoritmo $\times$ Red). Se utilizará una secuencia de semillas deterministas (1000 a 1014) inyectadas en `numpy.random` para garantizar la reproducibilidad exacta del experimento.

3. **Métrica Principal de Evaluación:**
   $$Z = \frac{B_{max}}{\text{Arcos Únicos Inspeccionados}}$$

In [19]:
import os
import gc
import json
import numpy as np
import dataclasses
import pandas as pd
from tqdm.notebook import tqdm
from scipy.stats import mannwhitneyu

In [20]:
import sys
from pathlib import Path

src_path = Path().resolve().parent / "src"
sys.path.append(str(src_path))

from aco import ACO_CARP
from tabu import TabuSearch_CARP
from graph_model import RedTuberias

In [21]:
# extracción de parámetros óptimos obtenidos en la Fase 1

parametros_aco = pd.read_csv("../data/historial_optuna_aco.csv")
parametros_tabu = pd.read_csv("../data/historial_optuna_tabu.csv")

best_params_aco = parametros_aco.loc[parametros_aco["value"].idxmin()].to_dict()
best_params_tabu = parametros_tabu.loc[parametros_tabu["value"].idxmin()].to_dict()

In [22]:
PARAM_ACO = {
    "alpha": best_params_aco["params_alpha"],
    "beta": best_params_aco["params_beta"],
    "rho": best_params_aco["params_rho"],
    "omega": best_params_aco["params_omega"],
    "n_hormigas": 20,
    "max_iter": 200
}

PARAM_TABU = {
    "t_vecindad": best_params_tabu["params_t_vecindad"],
    "tenencia_tabu": best_params_tabu["params_tenencia_tabu"],
    "omega": best_params_tabu["params_omega"],
    "max_iter": 500
}

In [23]:
BATERIA_MAX = 15000.0
N_CORRIDAS = 15

RUTA_RESULTADOS = "../data/resultados_fase2.jsonl"

INSTANCIAS = [
    "red_base_5x5.json",
    "red_densa_10x10.json",
    "red_realista_asim.json"
]

In [24]:
# función para guardar resultados con metadatos de la instancia

def guardar_resultado(resultado, instancia_nombre, filepath="./data/resultados_fase2.jsonl"):
    
    data = dataclasses.asdict(resultado)
    data["parametros"]["instancia_red"] = instancia_nombre
    
    with open(filepath, 'a', encoding='utf-8') as f:
        f.write(json.dumps(data) + '\n')

---

## 4. Ejecución Computacional

En la siguiente celda se detonará el bucle de experimentación masiva. El proceso generará un total de 90 ejecuciones (3 redes $\times$ 2 algoritmos $\times$ 15 corridas).

In [25]:
# reinicio del dataset de resultados
if os.path.exists(RUTA_RESULTADOS):
    os.remove(RUTA_RESULTADOS)

In [26]:
for archivo_instancia in tqdm(INSTANCIAS, desc="Evaluación de Topologías"):
    ruta_instancia = os.path.join("../data", archivo_instancia)
    
    if not os.path.exists(ruta_instancia):
        print(f"Error estructural: No se halló {archivo_instancia}. Verifique graph_generator.py")
        continue

    for i in tqdm(range(N_CORRIDAS), desc=f"Corridas ({archivo_instancia})", leave=False):
        semilla_control = 1000 + i 
        
        # TABU SEARCH:
        np.random.seed(semilla_control)
        red_tabu = RedTuberias.cargar_red(ruta_instancia)
        ts = TabuSearch_CARP(
            red_tuberias=red_tabu,
            bateria_maxima=BATERIA_MAX,
            **PARAM_TABU
        )
        guardar_resultado(ts.run(), archivo_instancia, RUTA_RESULTADOS)
        del red_tabu, ts
        
        # ACO:
        np.random.seed(semilla_control)
        red_aco = RedTuberias.cargar_red(ruta_instancia)
        aco = ACO_CARP(
            red=red_aco,
            bateria_max=BATERIA_MAX,
            Q=1.0,
            **PARAM_ACO
        )
        guardar_resultado(aco.run(), archivo_instancia, RUTA_RESULTADOS)
        del red_aco, aco

        gc.collect()    # limpieza de memoria

print("Experimentación concluida. Dataset generado.")

Evaluación de Topologías:   0%|          | 0/3 [00:00<?, ?it/s]

Corridas (red_base_5x5.json):   0%|          | 0/15 [00:00<?, ?it/s]

--- Iniciando Tabu Search ---
Iteraciones: 500 | Tenencia Tabú: 23
Batería Máxima: 15000.0 | Factor de Castigo: 675.2728832721965

Iteración 1 -> Nuevo Óptimo | Z: 405.41 | Cobertura: 37 tramos | Batería consumida: 14960.00
Iteración 27 -> Nuevo Óptimo | Z: 394.74 | Cobertura: 38 tramos | Batería consumida: 14610.00

Convergencia temprana en iteración 127.

--- Resultados Finales ---
Mejor Puntaje Z (Costo por tramo): 394.74
Tramos únicos inspeccionados: 38 / 40
Batería consumida total: 14610.00
Batería en deadheading: 3750.00
Tiempo de cómputo: 3.7737 seg
Ruta propuesta:
[0, 1, 2, 7, 8, 13, 12, 7, 6, 1, 2, 3, 8, 9, 4, 3, 2, 7, 12, 17, 22, 23, 24, 19, 14, 9, 8, 7, 6, 5, 6, 11, 16, 21, 22, 23, 18, 19, 24, 19, 18, 17, 16, 15, 20, 21, 16, 15, 10, 5, 0, 5, 10, 11, 12]
--- Iniciando ACO ---
Población: 20 hormigas | Iteraciones: 200
Batería Máxima: 15000.0 | Factor de Castigo: 1113.174580632946

Iteración 1 -> Nuevo Óptimo | Z: 375.00 | Cobertura: 40 tramos | Batería consumida: 14830.00

---

Corridas (red_densa_10x10.json):   0%|          | 0/15 [00:00<?, ?it/s]

--- Iniciando Tabu Search ---
Iteraciones: 500 | Tenencia Tabú: 23
Batería Máxima: 15000.0 | Factor de Castigo: 675.2728832721965

Iteración 1 -> Nuevo Óptimo | Z: 714.29 | Cobertura: 21 tramos | Batería consumida: 14974.00
Iteración 2 -> Nuevo Óptimo | Z: 625.00 | Cobertura: 24 tramos | Batería consumida: 14898.00
Iteración 4 -> Nuevo Óptimo | Z: 600.00 | Cobertura: 25 tramos | Batería consumida: 14598.00
Iteración 5 -> Nuevo Óptimo | Z: 576.92 | Cobertura: 26 tramos | Batería consumida: 14848.00
Iteración 16 -> Nuevo Óptimo | Z: 535.71 | Cobertura: 28 tramos | Batería consumida: 14736.00
Iteración 17 -> Nuevo Óptimo | Z: 468.75 | Cobertura: 32 tramos | Batería consumida: 14997.00
Iteración 52 -> Nuevo Óptimo | Z: 454.55 | Cobertura: 33 tramos | Batería consumida: 14844.00

Convergencia temprana en iteración 152.

--- Resultados Finales ---
Mejor Puntaje Z (Costo por tramo): 454.55
Tramos únicos inspeccionados: 33 / 180
Batería consumida total: 14844.00
Batería en deadheading: 150.00


Corridas (red_realista_asim.json):   0%|          | 0/15 [00:00<?, ?it/s]

--- Iniciando Tabu Search ---
Iteraciones: 500 | Tenencia Tabú: 23
Batería Máxima: 15000.0 | Factor de Castigo: 675.2728832721965

Iteración 1 -> Nuevo Óptimo | Z: 483.87 | Cobertura: 31 tramos | Batería consumida: 14974.00
Iteración 2 -> Nuevo Óptimo | Z: 454.55 | Cobertura: 33 tramos | Batería consumida: 14884.00
Iteración 29 -> Nuevo Óptimo | Z: 441.18 | Cobertura: 34 tramos | Batería consumida: 14979.00
Iteración 63 -> Nuevo Óptimo | Z: 428.57 | Cobertura: 35 tramos | Batería consumida: 14914.00

Convergencia temprana en iteración 163.

--- Resultados Finales ---
Mejor Puntaje Z (Costo por tramo): 428.57
Tramos únicos inspeccionados: 35 / 45
Batería consumida total: 14914.00
Batería en deadheading: 1420.00
Tiempo de cómputo: 2.6271 seg
Ruta propuesta:
[0, 28, 0, 5, 10, 15, 10, 11, 16, 17, 16, 15, 20, 21, 16, 29, 16, 11, 6, 5, 0, 1, 6, 7, 12, 13, 14, 19, 24, 23, 22, 17, 18, 13, 8, 3, 2, 7, 8, 9, 4, 3]
--- Iniciando ACO ---
Población: 20 hormigas | Iteraciones: 200
Batería Máxima: 15

---

## 5. Recolección de Datos y Primera Visualización

Una vez concluida la simulación, procesamos el archivo JSONL mediante Pandas para tabular los promedios y medianas. Esta estructura tabular servirá de insumo para la redacción del informe final y para las pruebas estadísticas de contraste de hipótesis.

In [27]:
registros = []
with open(RUTA_RESULTADOS, 'r', encoding='utf-8') as f:
    for line in f:
        registros.append(json.loads(line))

In [28]:
df_resultados = pd.json_normalize(registros)

# extraenis las variables de interés
df_analisis = df_resultados[[
    'algoritmo', 
    'parametros.instancia_red', 
    'fitness_final', 
    'arcos_unicos_inspeccionados', 
    'bateria_consumida_total',
    'bateria_consumida_deadheading',
    'tiempo_ejecucion_seg'
]]

In [29]:
# Tabla comparativa de métricas centrales agrupadas por Red y Algoritmo

tabla_resumen = df_analisis.groupby(['parametros.instancia_red', 'algoritmo']).agg(
    Mediana_Fitness=('fitness_final', 'median'),
    Max_Cobertura_Arcos=('arcos_unicos_inspeccionados', 'max'),
    Promedio_Cobertura_Arcos=('arcos_unicos_inspeccionados', 'mean'),
    Bateria_Consumida=('bateria_consumida_total', 'mean'),
).reset_index()

display(tabla_resumen)

,parametros.instancia_red,algoritmo,Mediana_Fitness,Max_Cobertura_Arcos,Promedio_Cobertura_Arcos,Bateria_Consumida
0,red_base_5x5.json,ACO,375.000000,40,40.000000,14883.333333
1,red_base_5x5.json,Tabu Search,375.000000,40,39.466667,14836.000000
2,red_densa_10x10.json,ACO,416.666667,36,35.866667,14902.533333
3,red_densa_10x10.json,Tabu Search,468.750000,35,32.133333,14924.666667
4,red_realista_asim.json,ACO,416.666667,36,36.000000,14916.000000
5,red_realista_asim.json,Tabu Search,428.571429,35,34.533333,14932.666667


---

## 6. Contraste de Hipótesis (Test U de Mann-Whitney)

Debido a que las metaheurísticas estocásticas no garantizan distribuciones normales en sus resultados (especialmente por la tendencia a agruparse en torno a óptimos locales, lo que genera asimetría), los supuestos de la prueba T de Student se violan.

Por lo tanto, generalmente se aplica el **Test U de Mann-Whitney**, una prueba no paramétrica robusta que compara las medianas de dos muestras independientes.

### Definición de Hipótesis Estadísticas:
* $H_0$ (Hipótesis Nula): No existe diferencia estadísticamente significativa entre las medianas del fitness de ACO y Tabu Search ($\tilde{Z}_{ACO} \ge \tilde{Z}_{TS}$).
* $H_1$ (Hipótesis Alternativa): El fitness mediano de ACO es estrictamente menor al de Tabu Search ($\tilde{Z}_{ACO} < \tilde{Z}_{TS}$).

Se establece un nivel de significancia $\alpha = 0.05$. Un valor $p < \alpha$ justificará el rechazo de la hipótesis nula, aportando validez científica a la dominancia del modelo propuesto.

In [30]:
# agrupamos por cada instancia topológica evaluada

for red in df_analisis['parametros.instancia_red'].unique():
    data_red = df_analisis[df_analisis['parametros.instancia_red'] == red]
    
    # vectores de muestras independientes
    fitness_aco = data_red[data_red['algoritmo'] == 'ACO']['fitness_final']
    fitness_tabu = data_red[data_red['algoritmo'] == 'Tabu Search']['fitness_final']
    
    # queremos probar que ACO es MENOR que Tabu
    stat, p_value = mannwhitneyu(fitness_aco, fitness_tabu, alternative='less')
    
    print(f"Red Evaluada: {red}")
    print(f"  Mediana ACO : {fitness_aco.median():.2f}")
    print(f"  Mediana Tabu: {fitness_tabu.median():.2f}")
    print(f"  Estadístico U: {stat} | p-valor: {p_value:.5e}")
    
    if p_value < 0.05:
        print("  Conclusión: Diferencia significativa a favor de ACO (Rechazo H0).")
    else:
        print("  Conclusión: Sin evidencia para afirmar superioridad estadística.")
    print("-" * 60)

Red Evaluada: red_base_5x5.json
  Mediana ACO : 375.00
  Mediana Tabu: 375.00
  Estadístico U: 60.0 | p-valor: 1.69553e-03
  Conclusión: Diferencia significativa a favor de ACO (Rechazo H0).
------------------------------------------------------------
Red Evaluada: red_densa_10x10.json
  Mediana ACO : 416.67
  Mediana Tabu: 468.75
  Estadístico U: 1.0 | p-valor: 7.38606e-07
  Conclusión: Diferencia significativa a favor de ACO (Rechazo H0).
------------------------------------------------------------
Red Evaluada: red_realista_asim.json
  Mediana ACO : 416.67
  Mediana Tabu: 428.57
  Estadístico U: 0.0 | p-valor: 2.14222e-07
  Conclusión: Diferencia significativa a favor de ACO (Rechazo H0).
------------------------------------------------------------


---

## 7. Visualización Empírica (Distribución de Soluciones)

Para complementar la inferencia estadística, generaremos Boxplots. Esta técnica de análisis exploratorio nos permitirá visualizar:
1. La dispersión estocástica intrínseca de cada algoritmo (rango intercuartílico).
2. La vulnerabilidad a óptimos locales (observando la longitud de los "bigotes" y valores atípicos).
3. La degradación algorítmica conforme la densidad topológica de la red aumenta (comparando la base $5\times5$ contra la densa $10\times10$).

In [31]:
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

df_agrupado = df_analisis.groupby(['parametros.instancia_red', 'algoritmo']).agg(
    mediana_arcos=('arcos_unicos_inspeccionados', 'median'),
    mediana_fitness=('fitness_final', 'median')
).reset_index()

# Gráfico 1: Barras de Cobertura
fig_bar = px.bar(
    df_agrupado,
    x="parametros.instancia_red",
    y="mediana_arcos",
    color="algoritmo",
    barmode="group",
    text_auto=True,
    title="Comparativa de Rendimiento: Mediana de Cobertura por Red",
    labels={
        "parametros.instancia_red": "Instancia Topológica",
        "mediana_arcos": "Arcos Únicos Inspeccionados (Mediana)",
        "algoritmo": "Algoritmo"
    },
    color_discrete_map={"ACO": "#1f77b4", "Tabu Search": "#d62728"}
)

fig_bar.update_traces(textfont_size=14, textangle=0, textposition="outside", cliponaxis=False)
fig_bar.update_layout(template="plotly_white", yaxis_title="Cantidad de Arcos [Mayor es mejor]")
fig_bar.show()

# Gráfico 2: Strip Plot del Fitness (Distribución y Robustez)
fig_strip = px.strip(
    df_analisis,
    x="parametros.instancia_red",
    y="fitness_final",
    color="algoritmo",
    stripmode="group",
    title="Distribución de Resultados y Robustez Algorítmica (15 corridas por red)",
    labels={
        "parametros.instancia_red": "Instancia Topológica",
        "fitness_final": "Fitness Final (Z) [Menor es mejor]",
        "algoritmo": "Algoritmo"
    },
    color_discrete_map={"ACO": "#1f77b4", "Tabu Search": "#d62728"}
)

fig_strip.update_traces(jitter=0.2, marker=dict(size=8, opacity=0.7, line=dict(width=1, color='DarkSlateGrey')))
fig_strip.update_layout(template="plotly_white")
fig_strip.show()